# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Dataset metadata: `to_json()` gives the JSON-LD dict
metadata = dataset.metadata.to_json()

# Print dataset title and description
print("{}: {}".format(
    getattr(dataset.metadata, 'name', ''),
    getattr(dataset.metadata, 'description', '')
))

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Fetch all record set summaries
print("RecordSets found in the metadata (access by '@id'):")
record_sets = []
if hasattr(dataset.metadata, 'record_sets'):
    # mlcroissant v1.1+
    rs_list = dataset.metadata.record_sets
elif hasattr(dataset.metadata, 'recordSet'):
    rs_list = dataset.metadata.recordSet
else:
    rs_list = []

for rs in rs_list:
    # Each rs is an mlcroissant.Field or dict depending on loading
    # Get the @id and name
    rs_dict = rs if isinstance(rs, dict) else rs.to_json()  # ensure dict
    print("- @id:", rs_dict.get('@id'), "| Name:", rs_dict.get('name', ""))
    record_sets.append(rs_dict.get('@id'))
    # For each record set, print fields
    if 'field' in rs_dict:
        print("  Fields in record set:")
        for field in rs_dict['field']:
            if isinstance(field, dict):
                field_id = field.get('@id')
                field_name = field.get('name', "")
            else:
                fdict = field.to_json() if hasattr(field, 'to_json') else {}
                field_id = fdict.get('@id')
                field_name = fdict.get('name', "")
            print("    - @id:", field_id, "| Name:", field_name)
    print()

# If no record sets found
if not record_sets:
    print("No record sets found in metadata. Check the Croissant schema.")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# You can specify the '@id' of each record set you want to extract
# If you know the record set ids, you may set them here. Let's auto-detect from previous cell:

if not record_sets:
    # You may need to hard-code or inspect the Croissant file
    print("Could not automatically obtain record sets. Please check the Croissant schema.")
else:
    print(f"Record sets available: {record_sets}")

dataframes = {}

for record_set_id in record_sets:
    print(f"Loading record set: {record_set_id}")
    try:
        records_iter = dataset.records(record_set=record_set_id)
        records = list(records_iter)
        if records:
            df = pd.DataFrame(records)
            dataframes[record_set_id] = df
            print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")
        else:
            print(f"No records found in record set '@id': {record_set_id}")
    except Exception as e:
        print(f"Error loading records for {record_set_id}: {e}")

# Show an example DataFrame's columns and head
if dataframes:
    first_rs = list(dataframes.keys())[0]
    print(f"Columns in record set '@id': {first_rs}")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# For this example, let's pick a record set and a numeric field to analyze.
# You can replace these values with those from your dataset overview.
import numpy as np

# Pick the first record set as an example
example_record_set_id = None
numeric_field = None
group_field = None

if dataframes:
    example_record_set_id = list(dataframes.keys())[0]
    df = dataframes[example_record_set_id]

    # Attempt to automatically select numeric fields
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field = numeric_fields[0]  # Example: take the first numeric field
        print(f"Selected numeric_field = {numeric_field}")
    else:
        print("No numeric fields found in the DataFrame. Please check the columns.")

    # Try to pick a group field (non-numeric)
    candidate_group_fields = df.select_dtypes(include=[object]).columns.tolist()
    if candidate_group_fields:
        group_field = candidate_group_fields[0]
        print(f"Selected group_field = {group_field}")

    if numeric_field:
        threshold = df[numeric_field].mean() if not np.isnan(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        if group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field} (means):")
            display(grouped_df.head())
    else:
        print("No numeric_field selected; skipping filtering and grouping.")
else:
    print("No dataframes available to perform EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Draw example distributions if we have a numeric field
if dataframes and numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    # If group_field exists, show avg of numeric_field by group_field
    if group_field and group_field in df.columns:
        group_means = df.groupby(group_field)[numeric_field].mean().sort_values().dropna()
        plt.figure(figsize=(10, 6))
        group_means.plot(kind='bar')
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.ylabel(f"Mean {numeric_field}")
        plt.xlabel(group_field)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

<!--
This notebook demonstrated how to load metadata, explore record sets and fields (referencing all by `@id`), and extract, process, and visualize data from a Croissant-formatted dataset using the `mlcroissant` library. Adapt sections as needed to analyze all record sets and specific fields relevant to your scientific questions.
-->